# Supplementary national sources: sector gap analysis and gap-filling

**Project:** The Power-Hungry World (CISC7204, Team 03)
**Measure:** final electricity consumption by sector, in TWh, 1990–2022
**Main source:** UNdata Energy Balance (`Datasets/electricity_consumption_1990_2023.csv`, unit TJ; 1 TWh = 3,600 TJ = 3.6 PJ)

This notebook:
1. finds which countries have sector gaps in UNdata, ranked by how much electricity they consume;
2. checks whether those gaps actually lose consumption or only affect which sector it is assigned to;
3. documents every national source examined for each gap country (accepted or rejected), with the reason.

**Rules followed** (from the progress note): sector-level data only; final consumption, not generation or sales, unless a sales series is explicitly documented as a proxy; preserve original categories; never treat missing values as zero; record the URL, table and licence for every source.

**Sector mapping (UNdata):** Industry = B26_121, Transport = B40_122, Residential = B50_1231, Services/Other = B48_1232 + B49_1235 + B51_1234. The parent aggregate B47_123 is **never** added to its components.

In [1]:
import re, io, urllib.request
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 40)
DATA = Path('Datasets')
TJ_PER_TWH, PJ_PER_TWH = 3600, 3.6
YEARS = range(1990, 2023)
SECTORS = {'Industry': ['B26_121'], 'Transport': ['B40_122'], 'Residential': ['B50_1231'],
           'Services/Other': ['B48_1232', 'B49_1235', 'B51_1234']}
SECTOR_CODES = sum(SECTORS.values(), [])

def download(url, dest):
    """Download once; later runs reuse the saved file (keeps the notebook reproducible offline)."""
    dest = Path(dest)
    if not dest.exists():
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        dest.write_bytes(urllib.request.urlopen(req, timeout=120).read())
    return dest

## 1. UN country and area names

`REF_AREA` uses UN M49 numeric codes. Names come from the UN SDMX code list `CL_AREA`.

**Important:** the energy balance uses some *combined-territory* codes, so a lookup by country name can wrongly report a country as absent:

| Code | UN name | Not to be confused with |
|---|---|---|
| 251 | France-Monaco | 250 France |
| 382 | Italy-San Marino | 380 Italy |
| 579 | Norway-Svalbard and Jan Mayen Islands | 578 Norway |
| 757 | Switzerland-Liechtenstein | 756 Switzerland |

France is therefore **present** in UNdata (code 251), not absent as the progress note recorded.

In [2]:
cl = download('https://data.un.org/WS/rest/codelist/UNSD/CL_AREA/1.0', DATA / 'UN_CL_AREA_codelist.xml')
root = ET.parse(cl).getroot()
_lk = {c.get('id'): c.findtext('./{*}Name') for c in root.findall('.//{*}Code')}
def area_name(code):
    return _lk.get(str(code)) or _lk.get(str(code).zfill(3))

un = pd.read_csv(DATA / 'electricity_consumption_1990_2023.csv')
un = un[un.TIME_PERIOD.between(1990, 2022)].copy()
un['TWh'] = un.OBS_VALUE / TJ_PER_TWH
un['area'] = un.REF_AREA.map(area_name)
print('Unmatched codes:', sorted(set(un.loc[un.area.isna(), 'REF_AREA'])))
pd.DataFrame({c: [area_name(c), c in set(un.REF_AREA)] for c in [250, 251, 380, 382, 578, 579, 756, 757]},
             index=['UN name', 'in dataset']).T

Unmatched codes: []


,UN name,in dataset
250,France,False
251,France-Monaco,True
380,Italy,False
382,Italy-San Marino,True
578,Norway,False
579,Norway-Svalbard and Jan Mayen Islands,True
756,Switzerland,False
757,Switzerland-Liechtenstein,True


## 2. Sector gap table, ranked by 2022 final electricity consumption

In [3]:
fec = un[un.TRANSACTION == 'B25_FEC'].pivot_table(index='REF_AREA', columns='TIME_PERIOD', values='TWh')
fec22 = fec[2022]

def describe_missing(missing):
    if not missing: return 'complete'
    if len(missing) == len(YEARS): return 'ABSENT'
    contiguous = missing == list(range(missing[0], missing[-1] + 1))
    span = f'{missing[0]}-{missing[-1]}' if contiguous else ','.join(map(str, missing))
    return f'missing {len(missing)}: {span}'

rows = []
for code, g in un.groupby('REF_AREA'):
    row = {'code': code, 'area': area_name(code), 'FEC_2022_TWh': round(fec22.get(code, float('nan')), 1)}
    for sector, txs in SECTORS.items():
        present = set(g.loc[g.TRANSACTION.isin(txs), 'TIME_PERIOD'])
        row[sector] = describe_missing(sorted(set(YEARS) - present))
    rows.append(row)

gap = pd.DataFrame(rows).sort_values('FEC_2022_TWh', ascending=False)
gap['share_%'] = (100 * gap.FEC_2022_TWh / fec22.sum()).round(2)
gap['cum_%'] = gap['share_%'].cumsum().round(1)
gap['any_gap'] = gap[list(SECTORS)].ne('complete').any(axis=1)
gap.to_csv(DATA / 'undata_sector_gap_table_1990_2022.csv', index=False)
gap.head(30)

,code,area,FEC_2022_TWh,Industry,Transport,Residential,Services/Other,share_%,cum_%,any_gap
40,156,China,8199.6,complete,complete,complete,complete,32.88,32.9,False
209,840,United States of America,4006.2,complete,complete,complete,complete,16.07,49.0,False
87,356,India,1403.4,complete,complete,complete,complete,5.63,54.6,False
96,392,Japan,902.0,complete,complete,complete,complete,3.62,58.2,False
161,643,Russian Federation,822.8,missing 2: 1990-1991,missing 2: 1990-1991,missing 2: 1990-1991,missing 2: 1990-1991,3.30,61.5,True
22,76,Brazil,550.9,complete,complete,complete,complete,2.21,63.7,False
101,410,Republic of Korea,541.7,complete,complete,complete,complete,2.17,65.9,False
33,124,Canada,533.0,complete,complete,complete,complete,2.14,68.0,False
72,276,Germany,477.2,missing 1: 1990-1990,missing 1: 1990-1990,missing 1: 1990-1990,missing 1: 1990-1990,1.91,69.9,True
65,251,France-Monaco,415.0,complete,complete,complete,complete,1.66,71.6,False


## 3. Do missing sector rows lose consumption?

For each of the top 30 countries and each year, compare the sum of the six sector components with total final consumption (B25_FEC). If they match, a missing row means the country does not use that category (or the consumption is reported in another category). No electricity is lost from the total.

In [4]:
top30 = gap.head(30)
p = un[un.REF_AREA.isin(top30.code)].pivot_table(index=['REF_AREA', 'TIME_PERIOD'], columns='TRANSACTION', values='OBS_VALUE')
ratio = p.reindex(columns=SECTOR_CODES).sum(axis=1, min_count=1) / p['B25_FEC'] * 100
off = ratio[(ratio < 99) | (ratio > 101)]
print(f'Country-years checked: {ratio.notna().sum()}; outside 99-101%: {len(off)}')
print(f'Range of sector sum / FEC: {ratio.min():.2f}% to {ratio.max():.2f}%')

Country-years checked: 987; outside 99-101%: 0
Range of sector sum / FEC: 100.00% to 100.00%


**Result:** in every top-30 country-year, the sector components reconcile to total final consumption. So:
- missing **Services/Other components** (for example "other n.e.s." absent for Japan and Canada) are categories those countries don't use, and need no national source;
- missing **Transport** rows don't remove electricity from the total; they only affect which sector it is assigned to. They still can't be set to zero without evidence.

### Real gaps that remain (top 30 consumers)

In [5]:
gap_list = top30.loc[top30[['Industry', 'Transport', 'Residential']].ne('complete').any(axis=1),
                     ['area', 'FEC_2022_TWh', 'share_%', 'Industry', 'Transport', 'Residential']]
gap_list

,area,FEC_2022_TWh,share_%,Industry,Transport,Residential
161,Russian Federation,822.8,3.30,missing 2: 1990-1991,missing 2: 1990-1991,missing 2: 1990-1991
72,Germany,477.2,1.91,missing 1: 1990-1990,missing 1: 1990-1990,missing 1: 1990-1990
89,Iran (Islamic Republic of),316.0,1.27,complete,missing 9: 1990-1998,complete
172,Saudi Arabia,309.5,1.24,complete,ABSENT,complete
88,Indonesia,294.2,1.18,complete,"missing 19: 1990,1991,1992,1993,1994...",complete
179,Viet Nam,255.8,1.03,complete,missing 28: 1995-2022,complete
193,Thailand,208.2,0.83,complete,missing 9: 1990-1998,complete
206,Egypt,172.4,0.69,complete,missing 22: 1990-2011,complete
116,Malaysia,168.4,0.68,complete,missing 7: 1990-1996,complete
197,United Arab Emirates,134.6,0.54,complete,ABSENT,complete


Russia (and Kazakhstan, Ukraine) 1990–1991 are years before the USSR broke up: the countries did not yet report as separate states. These series start in 1992 and are **not** filled.

---
## 4. France: SDES compared with UNdata (France-Monaco, code 251)

UNdata is complete for France. The SDES series (`Chiffres clés de l'énergie 2026`, sheet P3-75) is kept as a **cross-check only**. It is climate-adjusted, which accounts for its larger differences in residential consumption.

In [6]:
def undata_4sectors(code):
    g = un[un.REF_AREA == code].pivot_table(index='TIME_PERIOD', columns='TRANSACTION', values='TWh')
    return pd.DataFrame({'Industry': g.get('B26_121'), 'Transport': g.get('B40_122'),
                         'Residential': g.get('B50_1231'),
                         'Services/Other': g.reindex(columns=SECTORS['Services/Other']).sum(axis=1, min_count=1),
                         'FEC': g.get('B25_FEC')})

fr_un = undata_4sectors(251).drop(columns='FEC').stack().rename('UNdata')
fr_sdes = (pd.read_csv(DATA / 'france_electricity_consumption_4sectors_1990_2022.csv')
           .set_index(['year', 'proposal_sector']).consumption_twh.rename('SDES'))
fr = pd.concat([fr_un, fr_sdes], axis=1).dropna()
fr['diff_%'] = 100 * (fr.SDES - fr.UNdata) / fr.UNdata
fr.groupby(level=1)['diff_%'].describe()[['mean', 'min', 'max']].round(2)

,mean,min,max
Industry,0.02,-1.04,1.01
Residential,-0.09,-6.96,4.27
Services/Other,0.02,-1.76,2.55
Transport,-0.03,-1.11,0.01


**Decision:** use UNdata (code 251) for France, consistent with every other country. Note in the report that the series includes Monaco (well under 1 TWh).

---
## 5. Germany 1990: AG Energiebilanzen (AGEB)

UNdata has no sector values for Germany in 1990.

- **Source:** AG Energiebilanzen e.V., *Evaluation Tables of the Energy Balance for Germany, 1990–2024* (EBD24e, January 2026; final data to 2024). AGEB compiles Germany's official national energy balance.
- **URL:** https://ag-energiebilanzen.de/wp-content/uploads/EBD24e_Auswertungstabellen_english.pdf
- **Tables:** 6.1 total, 6.2 mining and manufacturing, 6.3 private households, 6.4 trade/commerce/services (TCS), 6.6 transport; row "Electricity", unit PJ.
- **Licence:** the PDF has no licence statement and none was found on the website. **Verify before publishing the raw PDF to a public repository.**

The PDF tables have merged columns, so values are read by matching each number's horizontal position to the year header.

In [7]:
import pymupdf

ageb_pdf = download('https://ag-energiebilanzen.de/wp-content/uploads/EBD24e_Auswertungstabellen_english.pdf',
                    DATA / 'Germany_AGEB_Evaluation_Tables_1990_2024_EBD24e.pdf')
NUM = re.compile(r'^-?[\d,]+(\.\d+)?$')
YEAR = re.compile(r'^(19|20)\d\d$')

def page_lines(page):
    lines = {}
    for w in page.get_text('words'):
        y = round((w[1] + w[3]) / 2)
        key = next((k for k in lines if abs(k - y) <= 2), y)
        lines.setdefault(key, []).append(w)
    return [sorted(v, key=lambda w: w[0]) for _, v in sorted(lines.items())]

def first_row(page, label):
    """Values of the first row whose first word equals `label` (the PJ block), keyed by year."""
    lines = page_lines(page)
    header = next(l for l in lines if sum(bool(YEAR.match(w[4])) for w in l) >= 10)
    year_x = [((w[0] + w[2]) / 2, int(w[4])) for w in header if YEAR.match(w[4])]
    row = next(l for l in lines if l[0][4] == label)
    out = {}
    for w in row:
        if NUM.match(w[4]):
            x = (w[0] + w[2]) / 2
            out[min(year_x, key=lambda t: abs(t[0] - x))[1]] = float(w[4].replace(',', ''))
    return out

doc = pymupdf.open(ageb_pdf)
AGEB_TABLES = {20: ('6.1', 'Total final electricity consumption'),
               21: ('6.2', 'Mining, quarrying and manufacturing'),
               22: ('6.3', 'Private households'),
               23: ('6.4', 'Trade, commerce and services (TCS)'),
               25: ('6.6', 'Transport')}
recs = []
for pg, (tab, cat) in AGEB_TABLES.items():
    for y, v in first_row(doc[pg - 1], 'Electricity').items():
        recs.append({'country': 'Germany', 'year': y, 'ageb_category': cat, 'value_pj': v,
                     'value_twh': round(v / PJ_PER_TWH, 3), 'source_table': f'AGEB EBD24e Table {tab} (PDF p.{pg})',
                     'publication': 'AG Energiebilanzen, Evaluation Tables of the Energy Balance for Germany 1990-2024 (EBD24e, Jan 2026)',
                     'source_url': 'https://ag-energiebilanzen.de/wp-content/uploads/EBD24e_Auswertungstabellen_english.pdf',
                     'accessed': '2026-11-07', 'data_status': 'final'})
ageb = pd.DataFrame(recs).sort_values(['ageb_category', 'year'])
ageb.to_csv(DATA / 'germany_ageb_electricity_1990_2024_original.csv', index=False)

w = ageb.pivot(index='year', columns='ageb_category', values='value_pj')
check = w.drop(columns='Total final electricity consumption').sum(axis=1) - w['Total final electricity consumption']
print('Years:', w.index.min(), '-', w.index.max(), '| sector sum minus total (PJ): min', check.min(), 'max', check.max())

Years: 1990 - 2024 | sector sum minus total (PJ): min -1.0 max 1.0


### Comparison with UNdata, 1991–2022

This checks whether AGEB measures the same thing as UNdata before its 1990 value is used.

In [8]:
AGEB_TO_SECTOR = {'Mining, quarrying and manufacturing': 'Industry', 'Transport': 'Transport',
                  'Private households': 'Residential', 'Trade, commerce and services (TCS)': 'Services/Other',
                  'Total final electricity consumption': 'FEC'}
ag = (w / PJ_PER_TWH).rename(columns=AGEB_TO_SECTOR)
de_un = undata_4sectors(276)
diff = (100 * (ag - de_un) / de_un).loc[1991:2022]
diff.agg(['mean', 'min', 'max']).round(1)

ageb_category,FEC,Industry,Residential,Services/Other,Transport
mean,0.3,-3.6,0.2,7.5,-0.5
min,-1.6,-10.8,-0.2,-1.0,-7.1
max,2.3,-0.6,5.4,20.4,1.5


**Findings**
- Residential, Transport and total final consumption agree closely, so AGEB and UNdata measure the same thing.
- Industry and Services/Other split differently, especially in the early 1990s (about −10% and +13% in 1991). AGEB counts **construction** under TCS, while UNdata counts it under industry.

**1990 output.** Both versions are kept:
- `ageb_twh`: AGEB as published;
- `linked_estimate_twh`: AGEB 1990 × the sector's 1991 UNdata/AGEB ratio. This is an **estimate**, used to match UNdata's sector boundaries.

For Transport and Residential the two versions are effectively the same. For Industry and Services/Other the team must choose one and document the choice (rule 9).

In [9]:
un91 = de_un.loc[1991]
de90 = []
for cat, sector in AGEB_TO_SECTOR.items():
    if sector == 'FEC': continue
    r = un91[sector] / ag.loc[1991, sector]
    de90.append({'country': 'Germany', 'year': 1990, 'sector': sector, 'ageb_category': cat,
                 'ageb_twh': round(ag.loc[1990, sector], 2), 'undata_1991_twh': round(un91[sector], 2),
                 'ageb_1991_twh': round(ag.loc[1991, sector], 2), 'link_ratio_un_over_ageb_1991': round(r, 4),
                 'linked_estimate_twh': round(ag.loc[1990, sector] * r, 2),
                 'note': ('AGEB and UNdata agree in 1991 (within 1%); AGEB value usable directly' if abs(r - 1) < 0.01 else
                          'Sector boundary differs from UNdata (AGEB industry excludes construction, which is in TCS); linked value is an ESTIMATE')})
de90 = pd.DataFrame(de90)
de90.to_csv(DATA / 'germany_electricity_1990_4sectors.csv', index=False)
de90[['sector', 'ageb_twh', 'linked_estimate_twh', 'link_ratio_un_over_ageb_1991']]

,sector,ageb_twh,linked_estimate_twh,link_ratio_un_over_ageb_1991
0,Industry,207.78,229.88,1.1064
1,Transport,13.61,13.65,1.0026
2,Residential,117.22,117.16,0.9994
3,Services/Other,116.39,102.60,0.8816


---
## 6. Viet Nam: transport (UNdata missing 1995–2022)

UNdata reports transport electricity only for 1990–1994, and it is tiny (0.04–0.09 TWh, under 1% of final consumption).

**Sources examined**

| Source | Result |
|---|---|
| EVN (Vietnam Electricity) annual reports and press releases, *commercial electricity by component* | **Rejected.** These are **sales**, not final consumption, and the categories are agriculture/forestry/fishery; industry & construction; trade, hotels & restaurants; residential & administration; others. There is **no transport category**: urban rail would sit in "others". |
| Ministry of Industry and Trade / Institute of Energy energy statistics | **Not available** as an open official sector series. A UN Statistics Division presentation (2017) notes Viet Nam had no dedicated energy statistics survey; data come from three state corporations. |
| ERIA and Danish Energy Agency outlook reports | **Rejected.** These are modelling or outlook reports, not official statistical series. |

**Decision:** leave Viet Nam transport 1995–2022 **missing**; do not set it to zero. Total final consumption is unaffected (section 3). Viet Nam's first urban metro line opened in Hanoi in 2021, so transport electricity before then was very likely negligible. This is context only, not a value.

In [10]:
vn = undata_4sectors(704)
vn[['Transport', 'FEC']].assign(transport_share_pct=100 * vn.Transport / vn.FEC).loc[1990:1996].round(3)

,Transport,FEC,transport_share_pct
TIME_PERIOD,,,
1990,0.036,6.200,0.581
1991,0.038,6.550,0.580
1992,0.040,6.975,0.573
1993,0.045,7.833,0.574
1994,0.087,9.233,0.942
1995,NaN,11.198,NaN
1996,NaN,13.375,NaN


---
## 7. Indonesia: transport (UNdata missing 1990–2007 and 2016)

- **Source:** Ministry of Energy and Mineral Resources (ESDM), *Handbook of Energy & Economic Statistics of Indonesia* (HEESI), Table 5.4.1 *Energy Consumption in Transportation Sector (in Original Unit)*, column **Electricity (GWh)**.
  - HEESI **2012** (data 2000–2011): https://www.esdm.go.id/assets/media/content/content-handbook-of-energy-economic-statistics-of-indonesia-2012-dcexnjb.pdf (PDF pp. 59–60)
  - HEESI **2018 final edition** (data 2008–2018): https://www.esdm.go.id/assets/media/content/content-handbook-of-energy-and-economic-statistics-of-indonesia-2018-final-edition.pdf (PDF p. 46)
  - HEESI **2024** (data 2014–2024, already in `Datasets/`) gives the same 2016 value (223 GWh).
- **Measure:** final energy consumption of the transport sector, the same concept as UNdata B40_122 (electric rail).
- **Licence:** no open-licence statement found on the ESDM publications page. **Verify before publishing the raw PDFs.**

In [11]:
heesi12 = pymupdf.open(download('https://www.esdm.go.id/assets/media/content/content-handbook-of-energy-economic-statistics-of-indonesia-2012-dcexnjb.pdf',
                                DATA / 'Indonesia_HEESI_2012.pdf'))
heesi18 = pymupdf.open(download('https://www.esdm.go.id/assets/media/content/content-handbook-of-energy-and-economic-statistics-of-indonesia-2018-final-edition.pdf',
                                DATA / 'Indonesia_HEESI_2018_final.pdf'))
to_num = lambda s: float(s.replace(',', ''))

# HEESI 2012: years are on PDF p.59; the right-hand half of the table (8 columns, last = Electricity GWh) is on p.60
years12 = [int(y) for y in re.findall(r'^(20\d\d)$', heesi12[58].get_text(), flags=re.M)]
tail = heesi12[59].get_text().split('GWh', 1)[1].split()
assert len(tail) == 8 * len(years12), 'unexpected table layout'
e12 = {y: to_num(tail[8 * i + 7]) for i, y in enumerate(years12)}

# HEESI 2018: one block of tokens per year on PDF p.46; the last token of each block is Electricity (GWh)
tok = heesi18[45].get_text().split()
# a year starts a data row only when followed by a number (the page header also contains '2018')
pos = [i for i, t in enumerate(tok[:-1]) if re.fullmatch(r'20(0[89]|1\d)', t) and re.fullmatch(r'[\d,]+', tok[i + 1])]
e18 = {int(tok[a]): to_num(tok[b - 1]) for a, b in zip(pos, pos[1:] + [len(tok)])}

id_un = undata_4sectors(360).Transport * 1000  # GWh
cmp = pd.DataFrame({'HEESI_2012_GWh': pd.Series(e12), 'HEESI_2018_GWh': pd.Series(e18), 'UNdata_GWh': id_un.round(0)}).loc[2000:2018]
cmp

,HEESI_2012_GWh,HEESI_2018_GWh,UNdata_GWh
2000,44.0,NaN,NaN
2001,49.0,NaN,NaN
2002,53.0,NaN,NaN
2003,53.0,NaN,NaN
2004,55.0,NaN,NaN
2005,55.0,NaN,NaN
2006,67.0,NaN,NaN
2007,85.0,NaN,NaN
2008,81.0,81.0,81.0
2009,111.0,111.0,111.0


**Checks**
- HEESI 2012 and HEESI 2018 agree with each other where they overlap (2008–2011).
- UNdata matches HEESI **exactly** for 2008–2014, so UNdata's Indonesian transport series comes from HEESI. This is the strongest evidence that the definitions match.
- From 2015 UNdata departs from HEESI (UNdata has likely been revised; 2015: 257 vs 205 GWh). The 2016 fill therefore comes from a slightly different vintage than its UNdata neighbours. This is flagged.

**Filled:** 2000–2007 (HEESI 2012) and 2016 (HEESI 2018). **Still missing:** 1990–1999. Pre-2008 HEESI editions are not on the ESDM site, and the 2010 edition is a scanned image covering the same 2000s window.

In [12]:
fill = [{'year': y, 'value_gwh': e12[y], 'edition': 'HEESI 2012', 'table': 'Table 5.4.1, PDF pp.59-60'} for y in range(2000, 2008)]
fill.append({'year': 2016, 'value_gwh': e18[2016], 'edition': 'HEESI 2018 final edition', 'table': 'Table 5.4.1, PDF p.46'})
idn = pd.DataFrame(fill)
idn.insert(0, 'country', 'Indonesia'); idn.insert(2, 'sector', 'Transport')
idn['value_twh'] = idn.value_gwh / 1000
idn['source'] = 'Ministry of Energy and Mineral Resources (ESDM), Handbook of Energy & Economic Statistics of Indonesia'
idn['note'] = ['Fills UNdata gap; UNdata matches HEESI exactly 2008-2014'] * 8 + \
              ['Fills UNdata gap; UNdata diverges from HEESI from 2015 (vintage difference) - flag']
idn['accessed'] = '2026-11-07'
idn.to_csv(DATA / 'indonesia_transport_electricity_2000_2016_heesi.csv', index=False)
idn

,country,year,sector,value_gwh,edition,table,value_twh,source,note,accessed
0,Indonesia,2000,Transport,44.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.044,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
1,Indonesia,2001,Transport,49.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.049,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
2,Indonesia,2002,Transport,53.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.053,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
3,Indonesia,2003,Transport,53.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.053,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
4,Indonesia,2004,Transport,55.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.055,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
5,Indonesia,2005,Transport,55.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.055,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
6,Indonesia,2006,Transport,67.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.067,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
7,Indonesia,2007,Transport,85.0,HEESI 2012,"Table 5.4.1, PDF pp.59-60",0.085,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata matches HEE...,2026-11-07
8,Indonesia,2016,Transport,223.0,HEESI 2018 final edition,"Table 5.4.1, PDF p.46",0.223,Ministry of Energy and Mineral Resou...,Fills UNdata gap; UNdata diverges fr...,2026-11-07


---
## 8. Egypt: transport (UNdata missing 1990–2011)

- **Source examined:** Egyptian Electricity Holding Company (EEHC), Ministry of Electricity and Renewable Energy, *Annual Report 2018/2019*. http://www.moee.gov.eg/english_new/EEHC_Rep/2018-2019en.pdf, table "Development of Total Sold Energy According on All Voltages (GWh)" (PDF p. 85).
- CAPMAS (the statistics agency) catalogue pages could not be reached from this network.

In [13]:
eehc = pymupdf.open(download('http://www.moee.gov.eg/english_new/EEHC_Rep/2018-2019en.pdf',
                             DATA / 'Egypt_EEHC_Annual_Report_2018-2019.pdf'))
lines = [l.strip() for l in eehc[84].get_text().splitlines() if l.strip()]
cats = [l for l in lines if not re.fullmatch(r'[\d/]+', l) and l not in ('85',)]
print('Usage categories in EEHC sold-energy table:', cats[3:])

Usage categories in EEHC sold-energy table: ['Industries', 'Agriculture', 'Utilities', 'Public lighting', 'Government Entities', 'Residential', 'Commercial and Others', 'Interconnection& BOOTs', 'Energy tangible', 'exchange and Colonies', 'Grand total']


---
## 11. Saudi Arabia: transport (re-check)

The earlier GASTAT extraction is unchanged (residential, commercial, government, industrial, other incl. agriculture). Re-check on 2026-11-07:
- GASTAT *Electrical Energy Statistics* and *Energy Efficiency Statistics*: no transport or railway category.
- ECRA *Annual Statistical Booklet* (2017 edition): the mirror copy was refused (HTTP 403). ECRA's published category list (residential, commercial, government, industrial, agricultural, other) has no transport line.

**Decision:** still **unresolved**. Riyadh Metro (opened in late 2024) and the electric Haramain railway (2018) are counted inside other categories. Transport is not set to zero and not reallocated.

---
## 12. Thailand: transport (UNdata missing 1990–1998)

- UNdata reports transport as **0.0 TWh in 1999**, with positive values from 2000. That fits Bangkok's BTS Skytrain opening in December 1999.
- The official source would be DEDE (Department of Alternative Energy Development and Efficiency), *Energy Balance of Thailand* / *Energy Statistics of Thailand*. **DEDE's websites could not be reached**, and the parliament-library copy was refused (HTTP 403).
- EPPO's electricity tables are **sales by tariff class** (residential, small/medium/large general service, government, agricultural pumping…), with no transport category, so they are rejected.

**Decision:** Thailand transport 1990–1998 stays **missing**. It's very likely nil (no electrified urban rail before December 1999), but this isn't documented by a source for those years, so it is not set to zero.

---
## 13. Malaysia: transport (UNdata missing 1990–1996)

- **Source examined:** Energy Commission (Suruhanjaya Tenaga), *National Energy Balance 2022*, Figure 31 *Electricity Consumption by Sectors* (PDF p. 60). https://www.st.gov.my/sites/default/files/2026-02/National_Energy_Balance_2022.pdf
- It covers only **2005–2022**. The note says transport data have been collected directly from train operators since 2006.
- The 1990s should be in older NEB editions (for example NEB 2012, which covers 1990–2012). These are no longer on the Energy Commission website, and the Internet Archive copy was rate-limited (HTTP 429) during this session. The MyEnergyStats portal requires registration.
- Malaysia's KTM Komuter (1995) and STAR LRT (December 1996) are the first electric urban rail services, so the gap probably covers small values in 1995–1996 and nil before that.

**Decision:** **pending.** Retry the archived NEB 2012 later. The URL is in the source register.

In [14]:
my_neb = pymupdf.open(download('https://www.st.gov.my/sites/default/files/2026-02/National_Energy_Balance_2022.pdf',
                               DATA / 'Malaysia_National_Energy_Balance_2022.pdf'))
txt = re.sub(r'\s+', ' ', my_neb[59].get_text())
print(txt[:120], '...')
print('Earliest year in Figure 31:', min(int(y) for y in re.findall(r'\b(20\d\d)\b', txt.split('ktoe')[0])))

E N E R G Y C O M M I S S I O N . 60 Figure 31 : Electricity Consumption by Sectors 2005 1,395 2,172 3,371 5,000 - 2006  ...
Earliest year in Figure 31: 2005


---
## 14. Pakistan: industry 1990–1992 and transport 2017–2022

**Important:** for 1990–1992 UNdata has no industry row **and** its total final consumption (B25_FEC) also leaves industry out. FEC jumps from 21.6 TWh (1992) to 36.5 TWh (1993). Unlike the transport gaps, this gap **understates the national total**.

- **Source:** State Bank of Pakistan, *Handbook of Statistics on Pakistan Economy*, Table 2.3 *Energy Consumption*, part 3 *Electricity (GWh)*, columns Traction, Household, Commercial, Industrial, Agricultural, Street Lights, (unlabelled), Other Govt., Total; fiscal years (July–June) 1971-72 to 2019-20. The table compiles Pakistan Energy Yearbook figures.
  https://archive.sbp.org.pk/departments/stats/pakEconomy_HandBook/Chap-2.3.pdf (PDF pp. 5–6)

In [15]:
sbp = pymupdf.open(download('https://archive.sbp.org.pk/departments/stats/pakEconomy_HandBook/Chap-2.3.pdf',
                            DATA / 'Pakistan_SBP_Handbook_Ch2.3_Energy_Consumption.pdf'))
FY = re.compile(r'^(19|20)\d\d-\d\d$')
num = lambda s: 0.0 if s == '-' else float(s.replace(',', ''))
rows = []
for pg in (4, 5):                                   # PDF pp.5-6, parsed separately so page numbers don't leak into rows
    tok = sbp[pg].get_text().split()
    starts = [i for i, t in enumerate(tok) if FY.match(t)]
    for a, b in zip(starts, starts[1:] + [len(tok)]):
        vals = []
        for t in tok[a + 1:b]:
            if re.fullmatch(r'[\d,]+|-', t): vals.append(t)
            elif t == 'P': continue                  # provisional marker
            elif vals: break
        if len(vals) < 8: continue                   # 1970s rows have no traction column; not needed
        rows.append({'fiscal_year': tok[a], 'provisional': 'P' in tok[a + 1:a + 3],
                     'Traction': num(vals[0]), 'traction_reported_as_dash': vals[0] == '-',
                     'Household': num(vals[1]), 'Commercial': num(vals[2]), 'Industrial': num(vals[3]),
                     'Agricultural': num(vals[4]), 'Street Lights': num(vals[5]),
                     'Other (unlabelled)': sum(num(v) for v in vals[6:-2]), 'Other Govt.': num(vals[-2]), 'Total': num(vals[-1])})
pk = pd.DataFrame(rows)
comp_cols = ['Traction', 'Household', 'Commercial', 'Industrial', 'Agricultural', 'Street Lights', 'Other (unlabelled)', 'Other Govt.']
pk['components_minus_total'] = pk[comp_cols].sum(axis=1) - pk.Total
pk.to_csv(DATA / 'pakistan_sbp_electricity_by_group_original.csv', index=False)
print('Fiscal years:', pk.fiscal_year.iloc[0], 'to', pk.fiscal_year.iloc[-1])
pk.loc[pk.components_minus_total.abs() > 2, ['fiscal_year', 'provisional', 'components_minus_total']]

Fiscal years: 1980-81 to 2019-21


,fiscal_year,provisional,components_minus_total
11,1991-92,False,310.0
39,2019-20,False,-10001.0


Rows where the source's own components don't add up to its total: 1991-92 (+310, about the street-lighting value), 2019-20 (about −10,000) and the provisional, mislabelled "2019-21 P" partial-year row. The 2019-20 and provisional rows are used only with a flag (or not at all).

### Which fiscal year does each UNdata year correspond to?

Match each UNdata industry value to the SBP fiscal year with the same value (±1 GWh):

In [16]:
pk_un = un[un.REF_AREA == 586].pivot_table(index='TIME_PERIOD', columns='TRANSACTION', values='OBS_VALUE') / TJ_PER_TWH * 1000  # GWh
sbp_ind = pk[~pk.provisional].set_index('fiscal_year').Industrial
yr_map = {y: next((fy for fy, x in sbp_ind.items() if abs(x - v) <= 1), None) for y, v in pk_un.B26_121.dropna().items()}
pd.Series(yr_map, name='SBP fiscal year matching UNdata industry').to_frame().T

,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
SBP fiscal year matching UNdata industry,1992-93,1993-94,1994-95,1995-96,1996-97,1997-98,1998-99,1999-00,2000-01,2002-03,2003-04,2004-05,2005-06,2006-07,2007-08,2008-09,2009-10,2010-11,2011-12,2012-13,2013-14,2014-15,2015-16,2016-17,NaN,2018-19,NaN,NaN,NaN,NaN


**Year convention (derived above):**
- up to **2001**, UNdata year *Y* = fiscal year *(Y−1)–Y* (for example 1993 = 1992-93);
- from **2002**, UNdata year *Y* = fiscal year *Y–(Y+1)* (for example 2002 = 2002-03). FY 2001-02 does not appear in UNdata.
- 2017 and 2019–2023 have no exact match: UNdata holds revised or newer figures (for example 2017: UNdata 27,398 vs SBP 2017-18 27,468 GWh, −0.3%).

With that mapping, the other sectors also match exactly:

In [17]:
def fy_for(y):
    return f'{y-1}-{str(y)[2:]}' if y <= 2001 else f'{y}-{str(y+1)[2:]}'
s = pk[~pk.provisional].set_index('fiscal_year')
cmp = pd.DataFrame({y: {'Industry': s.Industrial.get(fy_for(y)) - pk_un.B26_121[y],
                        'Residential': s.Household.get(fy_for(y)) - pk_un.B50_1231[y],
                        'Agriculture': s.Agricultural.get(fy_for(y)) - pk_un.B48_1232[y],
                        'Transport': s.Traction.get(fy_for(y)) - pk_un.B40_122.get(y, float('nan'))}
                    for y in range(1993, 2017)}).T
print('SBP minus UNdata (GWh), 1993-2016:')
cmp.agg(['min', 'max']).round(1)

SBP minus UNdata (GWh), 1993-2016:


,Industry,Residential,Agriculture,Transport
min,-0.0,-0.0,-1.0,-0.0
max,0.0,0.0,0.0,0.0


In [18]:
chk = pd.DataFrame({'UN_FEC_GWh': pk_un.B25_FEC.loc[1990:1993],
                    'SBP_Industrial': [s.Industrial[fy_for(y)] for y in range(1990, 1994)],
                    'SBP_Total': [s.Total[fy_for(y)] for y in range(1990, 1994)]}, index=range(1990, 1994))
chk['UN_FEC + SBP industry (1990-92)'] = chk.UN_FEC_GWh + chk.SBP_Industrial.where(chk.index < 1993, 0)
chk.round(0)

,UN_FEC_GWh,SBP_Industrial,SBP_Total,UN_FEC + SBP industry (1990-92)
1990,18445.0,10324.0,28769.0,28769.0
1991,20305.0,11229.0,31534.0,31534.0
1992,21589.0,12289.0,33878.0,33878.0
1993,36492.0,13043.0,36492.0,36492.0


UNdata 1990–1992 FEC + SBP industry ≈ SBP total, which confirms that UNdata's 1990–1992 FEC leaves out industry. The 1991-92 difference is the source's own 310 GWh street-lighting discrepancy.

**Filled**
- **Industry 1990–1992**: SBP *Industrial*, FY 1989-90 to 1991-92 (early convention). UNdata FEC for these years must be **recomputed** (add industry); do not use the published 1990–1992 B25_FEC.
- **Transport 2017–2019**: SBP reports traction as "–" (nil) for FY 2017-18, 2018-19 and 2019-20 (late convention). UNdata already records SBP's "–" as **0** for 2012–2016, so these are recorded as **reported nil (0)**, flagged as such. 2019 additionally comes from a row that does not reconcile to its total.
- **Transport 2020–2022: still missing.** The SBP table ends at FY 2019-20, and NEPRA's *State of Industry Report 2021* (sales by category: domestic, commercial, industrial, agricultural, public lighting, bulk supply, others) has a *railway traction* tariff but **no traction sales line**. Lahore's Orange Line metro (opened October 2020) is counted inside another category.

In [19]:
fills = [dict(year=y, fiscal_year=fy_for(y), sector='Industry', value_gwh=s.Industrial[fy_for(y)],
              note='Fills UNdata gap; UNdata B25_FEC 1990-1992 excludes industry and must be recomputed') for y in (1990, 1991, 1992)]
fills += [dict(year=y, fiscal_year=fy_for(y), sector='Transport', value_gwh=s.Traction[fy_for(y)],
               note='Source reports "-" (nil) for traction; recorded as reported nil, consistent with UNdata 0 for 2012-2016'
                    + ('; source row does not reconcile to its total - use with caution' if y == 2019 else ''))
          for y in (2017, 2018, 2019)]
pk_fill = pd.DataFrame(fills)
assert pk_fill.loc[pk_fill.sector == 'Transport', 'fiscal_year'].map(s.traction_reported_as_dash).all()
pk_fill.insert(0, 'country', 'Pakistan')
pk_fill['value_twh'] = pk_fill.value_gwh / 1000
pk_fill['source'] = 'State Bank of Pakistan, Handbook of Statistics on Pakistan Economy, Table 2.3 (Electricity, GWh), PDF pp.5-6'
pk_fill['source_url'] = 'https://archive.sbp.org.pk/departments/stats/pakEconomy_HandBook/Chap-2.3.pdf'
pk_fill['accessed'] = '2026-11-07'
pk_fill.to_csv(DATA / 'pakistan_electricity_fills_sbp.csv', index=False)
pk_fill[['year', 'fiscal_year', 'sector', 'value_gwh', 'note']]

,year,fiscal_year,sector,value_gwh,note
0,1990,1989-90,Industry,10324.0,Fills UNdata gap; UNdata B25_FEC 199...
1,1991,1990-91,Industry,11229.0,Fills UNdata gap; UNdata B25_FEC 199...
2,1992,1991-92,Industry,12289.0,Fills UNdata gap; UNdata B25_FEC 199...
3,2017,2017-18,Transport,0.0,"Source reports ""-"" (nil) for tractio..."
4,2018,2018-19,Transport,0.0,"Source reports ""-"" (nil) for tractio..."
5,2019,2019-20,Transport,0.0,"Source reports ""-"" (nil) for tractio..."


In [20]:
nepra = pymupdf.open(download('https://www.nepra.org.pk/publications/State%20of%20Industry%20Reports/State%20of%20Industry%20Report%202021.pdf',
                              DATA / 'Pakistan_NEPRA_State_of_Industry_Report_2021.pdf'))
print(re.sub(r'\s+', ' ', nepra[184].get_text())[:260])

State of Industry Report 2021 176 TABLE 69 Category-wise Electricity Sold (GWh) DISCO Year Domestic Commercial Industrial Agricultural Public Lighting Bulk Supply Others Total PESCO 2016-17 4882.80 739.50 2131.50 83.10 13.24 579.70 2.19 8432.03 2017-18 4928.26


---
## 15. Source register

In [21]:
register = pd.DataFrame([
    ('France', 'All sectors', 'UNdata code 251 (France-Monaco)', 'Use UNdata', 'SDES kept as cross-check (climate-adjusted)', 'SDES open licence', ''),
    ('Germany', 'All sectors, 1990', 'AGEB Evaluation Tables EBD24e, Tables 6.1-6.6', 'Accepted', 'Transport/Residential direct; Industry/Services/Other boundary differs, linked estimate offered', 'Not verified', 'https://ag-energiebilanzen.de/wp-content/uploads/EBD24e_Auswertungstabellen_english.pdf'),
    ('Saudi Arabia', 'Transport, all years', 'GASTAT Electricity Statistics 2021 & 2023; ECRA booklet', 'Unresolved', 'No transport category; ECRA copy refused (403)', 'GASTAT reuse with attribution; MoE rights not verified', ''),
    ('Indonesia', 'Transport 1990-2007, 2016', 'ESDM HEESI 2012 / 2018 Table 5.4.1', 'Accepted (2000-2007, 2016)', '1990-1999 still missing', 'Not verified', 'https://esdm.go.id/en/publikasi/handbook-of-energy-economic-statistics-of-indonesia-heesi'),
    ('Viet Nam', 'Transport 1995-2022', 'EVN sales by component', 'Rejected', 'Sales; no transport category', '-', ''),
    ('Thailand', 'Transport 1990-1998', 'DEDE Energy Balance; EPPO electricity statistics', 'Not accessible / rejected', 'DEDE unreachable; EPPO = sales by tariff, no transport. UNdata reports 0.0 in 1999', '-', ''),
    ('Egypt', 'Transport 1990-2011', 'EEHC Annual Report 2018/19', 'Rejected', 'Sales, fiscal years, no transport category', '-', 'http://www.moee.gov.eg/english_new/EEHC_Rep/2018-2019en.pdf'),
    ('Malaysia', 'Transport 1990-1996', 'Energy Commission NEB 2022 (2005+); NEB 2012 (archived)', 'Pending', 'NEB 2022 starts 2005; archive rate-limited - retry', 'Not verified', 'https://web.archive.org/web/20250514183643/https://www.st.gov.my/en/contents/publications/statistics_electricity/National%20Energy%20Balance%202012.pdf'),
    ('UAE', 'Transport, all years', 'FCSC/MoEI; DEWA statistics', 'Rejected', 'No transport category; DEWA is Dubai only', '-', ''),
    ('Pakistan', 'Industry 1990-1992; Transport 2017-2022', 'SBP Handbook Table 2.3; NEPRA SIR 2021', 'Accepted (industry 1990-92; transport 2017-19 reported nil)', 'UNdata FEC 1990-92 excludes industry; UNdata year convention switches in 2002; transport 2020-22 still missing', 'Not verified', 'https://archive.sbp.org.pk/departments/stats/pakEconomy_HandBook/Chap-2.3.pdf'),
    ('Iran', 'Transport 1990-1998', 'Ministry of Energy energy balance', 'Not accessible', 'Sites unreachable; Persian only', '-', ''),
], columns=['country', 'gap', 'source', 'decision', 'reason / caveat', 'licence', 'url'])
register.to_csv(DATA / 'supplementary_source_register.csv', index=False)
register.drop(columns='url')

,country,gap,source,decision,reason / caveat,licence
0,France,All sectors,UNdata code 251 (France-Monaco),Use UNdata,SDES kept as cross-check (climate-ad...,SDES open licence
1,Germany,"All sectors, 1990","AGEB Evaluation Tables EBD24e, Table...",Accepted,Transport/Residential direct; Indust...,Not verified
2,Saudi Arabia,"Transport, all years",GASTAT Electricity Statistics 2021 &...,Unresolved,No transport category; ECRA copy ref...,GASTAT reuse with attribution; MoE r...
3,Indonesia,"Transport 1990-2007, 2016",ESDM HEESI 2012 / 2018 Table 5.4.1,"Accepted (2000-2007, 2016)",1990-1999 still missing,Not verified
4,Viet Nam,Transport 1995-2022,EVN sales by component,Rejected,Sales; no transport category,-
5,Thailand,Transport 1990-1998,DEDE Energy Balance; EPPO electricit...,Not accessible / rejected,DEDE unreachable; EPPO = sales by ta...,-
6,Egypt,Transport 1990-2011,EEHC Annual Report 2018/19,Rejected,"Sales, fiscal years, no transport ca...",-
7,Malaysia,Transport 1990-1996,Energy Commission NEB 2022 (2005+); ...,Pending,NEB 2022 starts 2005; archive rate-l...,Not verified
8,UAE,"Transport, all years",FCSC/MoEI; DEWA statistics,Rejected,No transport category; DEWA is Dubai...,-
9,Pakistan,Industry 1990-1992; Transport 2017-2022,SBP Handbook Table 2.3; NEPRA SIR 2021,Accepted (industry 1990-92; transpor...,UNdata FEC 1990-92 excludes industry...,Not verified
